In [1]:
%pip install requests beautifulsoup4 minio

Note: you may need to restart the kernel to use updated packages.


In [2]:
MINIO_ENDPOINT = "minio:9000"
ACCESS_KEY = "dataNLPmining-lab"
SECRET_KEY = "dataNLPmining-lab"
BUCKET_NAME = "raw-financial-data" 

In [5]:
import requests
from bs4 import BeautifulSoup
import io
from minio import Minio
from minio.error import S3Error
from urllib.parse import urljoin


# Khởi tạo client MinIO
minio_client = Minio(
    MINIO_ENDPOINT,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    secure=False  # Đổi thành True nếu server MinIO của bạn dùng HTTPS
)

# Kiểm tra và tạo Bucket nếu chưa tồn tại
try:
    if not minio_client.bucket_exists(BUCKET_NAME):
        minio_client.make_bucket(BUCKET_NAME)
        print(f"✅ Đã tạo bucket mới: {BUCKET_NAME}")
except S3Error as e:
    print(f"❌ Lỗi kết nối MinIO: {e}")

# ==========================================
# HÀM 1: TẢI VÀ ĐẨY PDF LÊN MINIO
# ==========================================
def download_pdf_to_minio(pdf_url, filename):
    """
    Tải file PDF báo cáo tài chính từ URL dưới dạng byte 
    và đẩy thẳng luồng dữ liệu (stream) lên MinIO.
    """
    print(f"Đang tải báo cáo PDF từ: {pdf_url}")
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        # stream=True giúp không tải toàn bộ file khổng lồ vào RAM cùng lúc (tuy nhiên requests.content vẫn load hết vào RAM)
        response = requests.get(pdf_url, headers=headers, stream=True, timeout=15)
        response.raise_for_status()
        
        # Đọc nội dung PDF thành định dạng bytes
        pdf_bytes = response.content
        pdf_stream = io.BytesIO(pdf_bytes)
        pdf_size = len(pdf_bytes)

        # Upload lên MinIO
        minio_client.put_object(
            bucket_name=BUCKET_NAME,
            object_name=filename,
            data=pdf_stream,
            length=pdf_size,
            content_type="application/pdf"
        )
        print(f"✅ Đã tải và lưu thành công lên MinIO: {filename} ({pdf_size / 1024 / 1024:.2f} MB)")

    except Exception as e:
        print(f"❌ Lỗi khi tải hoặc upload PDF ({filename}): {e}")

# ==========================================
# HÀM 2: CRAWL LINK PDF TỪ TRANG WEB
# ==========================================
def crawl_apple_reports(url):
    """
    Quét trang web để tìm các link tải báo cáo tài chính PDF 
    và gọi hàm download_pdf_to_minio.
    """
    print(f"Đang quét dữ liệu từ: {url}")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
    }
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # annualreports.com thường để link PDF trong các thẻ <a> với href chứa "/Click/" hoặc kết thúc bằng ".pdf"
        # Bạn có thể cần điều chỉnh selector này tùy theo cấu trúc HTML thực tế của trang web.
        pdf_links = soup.find_all('a', href=True)
        
        count = 0
        for link in pdf_links:
            href = link['href']
            # Kiểm tra xem link có phải là PDF báo cáo không
            if '.pdf' in href.lower() or '/click/' in href.lower():
                # Tạo URL hoàn chỉnh nếu link là dạng tương đối
                full_pdf_url = urljoin(url, href)
                
                # Tạo tên file linh hoạt dựa trên text của link hoặc đánh số
                link_text = link.text.strip().replace(" ", "_").replace("/", "-")
                if not link_text:
                    link_text = f"Apple_Report_{count}"
                    
                filename = f"Apple/{link_text}.pdf" 
                
                download_pdf_to_minio(full_pdf_url, filename)
                count += 1
                
        print(f"🎉 Hoàn thành! Đã xử lý {count} báo cáo.")

    except Exception as e:
        print(f"❌ Lỗi khi crawl dữ liệu trang web: {e}")

# ==========================================
# CHẠY THỬ NGHIỆM (TESTING)
# ==========================================
if __name__ == "__main__":
    apple_url = "https://www.annualreports.com/Company/apple-inc"
    crawl_apple_reports(apple_url)

Đang quét dữ liệu từ: https://www.annualreports.com/Company/apple-inc
❌ Lỗi khi crawl dữ liệu trang web: HTTPSConnectionPool(host='www.annualreports.com', port=443): Max retries exceeded with url: /Company/apple-inc (Caused by NewConnectionError("HTTPSConnection(host='www.annualreports.com', port=443): Failed to establish a new connection: [Errno 111] Connection refused"))
